# Figure panels for hit model fits across datasets

## Imports

Calcium tracks data processing functions are in the `analysis/tracks.jl` module. Functions that make final, detailed plots are in `analysis/tracksPlots.jl`. Functions to fit the different hit models are in `analysis/modelsML.jl`. 

In [ ]:
using Plots: Plots as pl
using Printf: @sprintf
using Measures: mm
using Optim: Optim, optimize, IPNewton


# Load the data processing module
module tracksAnalysis
    include("analysis/tracks.jl")
end
using .tracksAnalysis: prepalldsets, loadalldsets, threshalldsets, 
        prepare_all_cdfs, get_csv_file_names, cleannames

module modelsML
    include("analysis/modelsML.jl")
end
using .modelsML: onehit_cdf, onehit_loglhood_total,
    twohit_cdf, twohit_loglhood_total,
    threehit_cdf, threehit_loglhood_total

module tracksPlots
    include("analysis/tracksPlots.jl")
end
#using .tracksPlots: plotrobustness


In [ ]:
# Default styles
pl.default(
    labelfontsize = 14,
    tickfontsize = 12,
    framestyle = :axes,  # https://docs.juliaplots.org/stable/gallery/gr/generated/gr-ref034/
    yminorgrid = false,
    xminorgrid = false,
    xgrid = false,
    ygrid = false,
    size = (550, 450),
    linewidth = 2.0,
    legendfontsize=12, 
    fontfamily="Arial"
)

# Custom line styles with shorter dashes
ddash = [8, 4]
dotdash = [4, 4, 4]

# Default analysis parameters
default_fluo_factor = 0.8


# Default analysis parameters
default_fluo_factor = 0.8
cdfthreshold = 0.75
# Filtering parameters: maximal start time, minimal traj length
minlength = 50
maxstarttime = 30.0
filter_params = [minlength, maxstarttime]

# Fit each model across datasets

In [ ]:
### RMS ERROR COMPUTATION ###

"""
Compute the root-mean-squared error of the fitted cumulative distribution 
function across all datasets.

Arguments:
- p_all: fitted parameters array [k2, kc_1, kc_2, ..., kc_n, t0_1, t0_2, ..., t0_n]
  where k2 is shared across all datasets, while kc_n and t0_n
  are the hit rate and the activation time offset for dataset n.
- tu_all: list of unique time arrays for each dataset
- cumuls_all: list of empirical cumulative distributions for each dataset
- ilim0s_all: list of lower indices (first point with t > 0) for each dataset
- ilim1s_all: list of upper indices (last point to include) for each dataset
- concs_all: list of concentrations for each dataset
- cdffct: the CDF function to use (e.g., onehit_cdf, twohit_cdf)

Returns:
- rms_errors: the root-mean-squared error computed on each dataset
- overall_rms: the total RMS across all datasets
"""
function compute_rms_all_dsets(p_all, tu_all, cumuls_all, ilim0s_all, 
                                ilim1s_all, concs_all, cdffct)
    n_dsets = length(tu_all)
    rms_errors = zeros(Float64, n_dsets)
    sres_total = 0.0
    npts_total = 0
    
    k2 = p_all[1]
    
    for n in 1:n_dsets
        kc = p_all[1 + n]
        tu = tu_all[n]
        cumul = cumuls_all[n]
        ilim0 = ilim0s_all[n]
        ilim1 = ilim1s_all[n]
        conc = concs_all[n]
        t0 = p_all[1 + n_dsets + n]  # t0 for dataset n
        
        # Determine starting index for RMS calculation
        # Prevent issues if there is no point with t_activ > 0
        rms_i = (ilim1 > ilim0) ? ilim0 : 1
        
        # Compute the fitted CDF for this dataset
        cdfestim = cdffct(tu[rms_i:ilim1], kc, k2; t0=t0)
        
        # Accumulate squared residuals
        dataset_sres = sum((cumul[rms_i:ilim1] .- cdfestim).^2)
        dataset_npts = (ilim1 - rms_i + 1)
        rms_errors[n] = sqrt(dataset_sres / dataset_npts)

        sres_total += dataset_sres
        npts_total += dataset_npts
    end

    # Compute overall RMS error for the full fit across all datasets
    overall_rms = sqrt(sres_total / npts_total)
    return rms_errors, overall_rms
end

In [ ]:
### MAIN FITTING FUNCTIONS ###

function fitmodel(tactiv_alldsets, dsetnames, durations, cdfthresh, loglfct, cdffct)
    # kc estimate: diffusion-limited, 0.4 pM^-1 s^-1, 
    # times concentration with default unit scale of 1 pM. 
    # k2 estimate: fast to begin with
    lower = [1e-4]  # k2
    upper = [20.0]
    init_p = [0.1]
    
    # For each dataset in the list of datasets, prepare the cdf, 
    # numbers of cells, concentrations, and limits for the fit.
    res = prepare_all_cdfs(tactiv_alldsets, dsetnames, durations, cdfthresh)
    concs_all, ncell_all, k0_all, k1_all, te_all, tu_all, cumuls_all = res

    # Limits for parameter kc of each dataset
    n_dsets = length(tactiv_alldsets)
    for i in 1:1:n_dsets
        concs_all
        push!(lower, 1e-4 * concs_all[i])  # kc lower bound, times conc
        push!(upper, 20.0 * concs_all[i])  # kc upper bound, times conc
        push!(init_p, 0.4 * concs_all[i])  # kc initial guess, diffusion-limit
    end
    

    # Limits for parameter t0 for each dataset: first recorded time point
    for i in 1:1:n_dsets
        push!(lower, -20.0)
        push!(upper, tu_all[i][1]-1e-3)
        push!(init_p, -0.1)
    end


    # Minimize -log likelihood
    cost = p_all -> -loglfct(p_all, tu_all, concs_all, ncell_all, 
                        k0_all, k1_all; te_all=te_all)
    inner_optimizer = IPNewton()  # or NelderMead()
    #println("Start linear fit with data points: ", tu)
    
    results = optimize(cost,
                lower, upper, 
                init_p, inner_optimizer
            )
    #println("Finished maximum likelihood fit")

    # Get the results
    fit_params = Optim.minimizer(results)
    k2 = fit_params[1]
    kcs = fit_params[2:1+n_dsets]
    t0s = fit_params[2+n_dsets:end]
    flag = Optim.converged(results)
    if !flag
        println("Did not converge for cdfhresh = ", cdfthresh)
        println("k1_all = ", k1_all, " , ncells = ", ncell_all, " , init_p = ", init_p)
    end

    # RMS error on the cdf across all datasets
    rms_per_dset, rmserr = compute_rms_all_dsets(
        fit_params, tu_all, cumuls_all, 
        k0_all, k1_all, concs_all, cdffct)
    
    result = Array{Float64}(vcat(fit_params, [rmserr]))

    return result, rms_per_dset
end


In [ ]:
### PLOTTING FUNCTIONS ###

"""
Create a multi-panel plot comparing one-hit and two-hit models across all datasets.

Arguments:
- tactiv_alldsets: list of activation time dataframes
- nicenames: list of dataset names
- durations: list of experiment durations
- cdfthresh: CDF threshold used for fitting
- onehit_params: [k2, kc_1, kc_2, ..., kc_n, t0_1, t0_2, ..., t0_n]
- twohit_params: [k2, kc_1, kc_2, ..., kc_n, t0_1, t0_2, ..., t0_n]
- threehit_params: [k2, kc_1, kc_2, ..., kc_n, t0_1, t0_2, ..., t0_n]
- onehit_cdf, twohit_cdf, threehit_cdf: CDF functions
- onehit_rms, twohit_rms, threehit_rms: RMS errors for model comparison
"""
function plot_ml_cdf_comparison(cdf_prep_results, nicenames, 
                                 onehit_params, twohit_params, threehit_params,
                                 onehit_cdf, twohit_cdf, threehit_cdf,
                                 onehit_rms, twohit_rms, threehit_rms)
    (concs_all, ncell_all, k0_all, k1_all, 
                te_all, tu_all, cumuls_all) = cdf_prep_results
    n_dsets = length(tu_all)

    # Determine layout (try to make roughly square)
    nrows = ceil(Int, sqrt(n_dsets))
    ncols = ceil(Int, n_dsets / nrows)
    
    # Create subplot layout
    p = pl.plot(layout=(nrows, ncols), size=(400*ncols, 350*nrows),
                left_margin=3mm, right_margin=3mm, bottom_margin=3mm)
    tplotlim = [-5.0, maximum([maximum(a) for a in tu_all]) * 1.05]
    fracplotlim = [-0.05, 1.05]
    for i in 1:n_dsets
        
        tu = tu_all[i]
        cumul = cumuls_all[i]
        ilim1 = k1_all[i]
        ncells = ncell_all[i]
        tlim = te_all[i]
        ncells = 1  # plot normalized cdfs
        
        # Scale CDF for plotting
        goodt = vcat([0], tu)
        ncumul = vcat([0], cumul .* ncells)
        
        # Get parameters for this dataset
        k2_1, kc_1, t0_1 = onehit_params[1], onehit_params[1+i], onehit_params[1+n_dsets+i]
        k2_2, kc_2, t0_2 = twohit_params[1], twohit_params[1+i], twohit_params[1+n_dsets+i]
        k2_3, kc_3, t0_3 = threehit_params[1], threehit_params[1+i], threehit_params[1+n_dsets+i]
        
        # Generate smooth curves
        trange = collect(LinRange(0, tu[end], 100))
        
        cdf1 = onehit_cdf(trange, kc_1, k2_1; t0=t0_1)
        y1 = cdf1 .* ncells
        
        cdf2 = twohit_cdf(trange, kc_2, k2_2; t0=t0_2)
        y2 = cdf2 .* ncells
        
        cdf3 = threehit_cdf(trange, kc_3, k2_3; t0=t0_3)
        y3 = cdf3 .* ncells
        
        # Plot on subplot i
        pl.plot!(p[i], goodt, ncumul, linetype=:steppost,
                 color=:black, linewidth=2, label="Data", alpha=0.7, 
                 legend=:bottomright, legendfontsize=8, 
                 xlim=tplotlim, ylim=fracplotlim)
        
        pl.scatter!(p[i], tu, cumul .* ncells, color=:black,
                    markersize=3, label=nothing, alpha=0.5)
        
        pl.plot!(p[i], trange, y1, label="1-hit",
                 color=:blue, linewidth=1.8, linestyle=:solid)
        
        pl.plot!(p[i], trange, y2, label="2-hit",
                 color=:red, linewidth=1.8, linestyle=:dash)
        
        pl.plot!(p[i], trange, y3, label="3-hit",
                 color=:purple, linewidth=1.8, linestyle=:dot)
        
        pl.xlabel!(p[i], "Time (s)", fontsize=10)
        pl.ylabel!(p[i], "Fraction active cells", fontsize=10)
        pl.title!(p[i], nicenames[i], fontsize=11)
    end
    for i in n_dsets+1:1:nrows * ncols
        pl.plot!(p[i], [], [], axis=false, legend=false, grid=false)
    end
    
    # Add overall title with RMS comparison
    #title_text = @sprintf "ML Fits: %d dataset(s) | RMS: 1-hit=%.2e, 2-hit=%.2e" n_dsets onehit_rms twohit_rms
    #pl.plot!(title=title_text, plot_title_fontsize=12)
    
    return p
end



In [ ]:
### MAIN ANALYSIS FUNCTION ###

function fit_ml_all_datasets(; 
    datadir="data/JR064_tracks",
    plotdir="figures/final_tracks_panels/",
    fracabove=default_fluo_factor,
    cdfthresh=cdfthreshold)
    
    # Create output directory if it doesn't exist
    if !isdir(plotdir)
        mkpath(plotdir)
    end
    
    # Get a cleaned up list of data file names and paths
    fnames, fpaths = get_csv_file_names(datadir)
    
    println("\n" * "="^70)
    println("MAXIMUM LIKELIHOOD FITTING - JR064 DATASETS")
    println("="^70)
    println("\nLoading and preprocessing datasets...")
    
    # Step 1: Preprocess all datasets
    tactiv_alldsets, durations, nicenames = prepalldsets(fpaths; 
        fracabove=fracabove, disp=false, filterpms=filter_params)
    
    n_dsets = length(tactiv_alldsets)
    println("\nLoaded $n_dsets datasets:")
    for (i, name) in enumerate(nicenames)
        println("  [$i] $name")
    end
    
    # Step 2: Fit the one-hit model
    println("\n" * "-"^70)
    println("Fitting ONE-HIT model (overall k2, one kc and one t0 per dataset)...")
    println("-"^70)
    onehit_result, onehit_all_rms = fitmodel(tactiv_alldsets, 
                    nicenames, durations, cdfthresh, 
                    onehit_loglhood_total, onehit_cdf)
    onehit_params = onehit_result[1:end-1]
    onehit_k2 = onehit_params[1]
    onehit_kcs = onehit_result[2:1+n_dsets]
    onehit_t0s = onehit_result[2+n_dsets:end-1]
    onehit_rms = onehit_result[end]
    
    println("\nOne-hit model results:")
    println(@sprintf "  k2        = %.4f s^-1" onehit_k2)
    println("  t0 per dataset:")
    for (i, t0) in enumerate(onehit_t0s)
        println(@sprintf "    Dataset %d (%s):  kc = %.4f s^-1" i nicenames[i] onehit_kcs[i])
        println(@sprintf "    Dataset %d (%s): t0 = %.4f s" i nicenames[i] t0)
    end
    
    # Step 3: Fit the two-hit model
    println("\n" * "-"^70)
    println("Fitting TWO-HIT model (overall k2, one kc and one t0 per dataset)...")
    println("-"^70)
    twohit_result, twohit_all_rms = fitmodel(tactiv_alldsets, 
                        nicenames, durations, cdfthresh,
                        twohit_loglhood_total, twohit_cdf)
    twohit_params = twohit_result[1:end-1]
    twohit_k2 = twohit_params[1]
    twohit_kcs = twohit_result[2:1+n_dsets]
    twohit_t0s = twohit_result[2+n_dsets:end-1]
    twohit_rms = twohit_result[end]
    
    println("\nTwo-hit model results:")
    println(@sprintf "  k2        = %.4f s^-1" twohit_k2)
    println("  t0 per dataset:")
    for (i, t0) in enumerate(twohit_t0s)
        println(@sprintf "    Dataset %d (%s):  kc = %.4f s^-1" i nicenames[i] twohit_kcs[i])
        println(@sprintf "    Dataset %d (%s): t0 = %.4f s" i nicenames[i] t0)
    end
    
    # Step 4: Fit the three-hit model
    println("\n" * "-"^70)
    println("Fitting THREE-HIT model (overall k2, one kc and one t0 per dataset)...")
    println("-"^70)
    threehit_result, threehit_all_rms = fitmodel(tactiv_alldsets, 
                            nicenames, durations, cdfthresh,
                            threehit_loglhood_total, threehit_cdf)
    threehit_params = threehit_result[1:end-1]
    threehit_k2 = threehit_params[1]
    threehit_kcs = threehit_result[2:1+n_dsets]
    threehit_t0s = threehit_result[2+n_dsets:end-1]
    threehit_rms = threehit_result[end]
    
    println("\nThree-hit model results:")
    println(@sprintf "  k2        = %.4f s^-1" threehit_k2)
    println("  t0 per dataset:")
    for (i, t0) in enumerate(threehit_t0s)
        println(@sprintf "    Dataset %d (%s):  kc = %.4f s^-1" i nicenames[i] threehit_kcs[i])
        println(@sprintf "    Dataset %d (%s): t0 = %.4f s" i nicenames[i] t0)
    end

    # Step 5: Compute and compare RMS errors
    println("\n" * "="^70)
    println("RMS ERROR COMPARISON")
    println("="^70)

    println("RMS per dataset:")
    for (i, t0) in enumerate(threehit_t0s)
        println(@sprintf "\n    Dataset %d (%s), one-hit = %.6e" i nicenames[i] onehit_all_rms[i])
        println(@sprintf "    Dataset %d (%s), two-hit = %.6e" i nicenames[i] twohit_all_rms[i])
        println(@sprintf "    Dataset %d (%s), three-hit = %.6e" i nicenames[i] threehit_all_rms[i])
    end
    
    # Step 6: Create comparison plots
    println("\n" * "-"^70)
    println("Generating plots...")
    println("-"^70)

    # Prepare all empirical CDFs
    prep_cdf_res = prepare_all_cdfs(tactiv_alldsets, nicenames, durations, cdfthresh)
    
    # Plot: All datasets, both models
    p_all = plot_ml_cdf_comparison(prep_cdf_res, nicenames, 
                                    onehit_params, twohit_params, threehit_params,
                                    onehit_cdf, twohit_cdf, threehit_cdf,
                                    onehit_rms, twohit_rms, threehit_rms)
    plot_file = joinpath(plotdir, "ml_cdf_all_datasets_comparison.pdf")
    pl.savefig(p_all, plot_file)
    println("  Saved: $plot_file")
    display(p_all)

    # Plot: one dataset, for illustration

    println("\n" * "="^70)
    println("Analysis complete!")
    println("="^70 * "\n")
    
    return (onehit_params, twohit_params, threehit_params, 
            onehit_rms, twohit_rms, threehit_rms,
            nicenames, tactiv_alldsets)
end

In [ ]:
fit_ml_all_datasets();

# Robustness analysis
Wrap in a main function, define plotting functions above (improved versions of the PLots-based ones in the scripts). 

In [ ]:

# Robustness analysis
# as we vary the fluorescence threshold
# Main analysis + plotting function for varying the fluorescence threshold
function robustness_vs_thresh_main(; 
    datadir="data/JR064_tracks",
    plotdir="figures/final_tracks_panels/",
    fracabove_default=default_fluo_factor,
    cdfthresh=cdfthreshold, 
    dosave=true)
    
    # Create output directory if it doesn't exist
    if !isdir(plotdir)
        mkpath(plotdir)
    end
    
    # Get a cleaned up list of data file names and paths
    fnames, fpaths = get_csv_file_names(datadir)

    nfluopts = 50
    
    # Save CDF fits for each file and each threshold fraction tried
    # number of parameters per dataset: one k2 plus (kc, t0) for each dset
    n_dsets = length(fnames)
    all_p1 = zeros(Float64, nfluopts, 1 + 2*n_dsets)
    all_rms1 = zeros(Float64, nfluopts, n_dsets)
    all_p2 = zeros(Float64, nfluopts, 1 + 2*n_dsets)
    all_rms2 = zeros(Float64, nfluopts, n_dsets)
    all_p3 = zeros(Float64, nfluopts, 1 + 2*n_dsets)
    all_rms3 = zeros(Float64, nfluopts, n_dsets)
    
    fluothreshrange = collect(LinRange(0.1, 1.5, nfluopts))

    # Load data once
    tracks_alldsets, nicenames = loadalldsets(fpaths; disp=false)

    # Repeat analysis for a range of fluorescence thresholds
    for j in 1:1:nfluopts
        tactiv_alldsets, durations = threshalldsets(
            tracks_alldsets, nicenames, fluothreshrange[j]; disp=false)
    
        # Fit the one-hit, two-hit, and three-hit models
        p1, rms1 = fitmodel(tactiv_alldsets, nicenames, durations, cdfthresh, 
                    onehit_loglhood_total, onehit_cdf)
        all_p1[j, :] = p1[1:end-1]  # last entry is RMS
        all_rms1[j, :] = rms1

        p2, rms2 = fitmodel(tactiv_alldsets, nicenames, durations, cdfthresh, 
                    twohit_loglhood_total, twohit_cdf)
        all_p2[j, :] = p2[1:end-1]
        all_rms2[j, :] = rms2

        p3, rms3 = fitmodel(tactiv_alldsets, nicenames, durations, cdfthresh, 
                    threehit_loglhood_total, threehit_cdf)
        all_p3[j, :] = p3[1:end-1]
        all_rms3[j, :] = rms3
        println(@sprintf "Finished fits for fluorescence threshold factor = %.2f" fluothreshrange[j]+1.0)
    end

    # Plot slope and goodness of linear vs quadratic fits
    # as a function of the threshold fraction
    # linfits, quadfits, allnames, fracrange, fname
    all_params = (all_p1, all_p2, all_p3)
    all_rms = (all_rms1, all_rms2, all_rms3)
    return all_params, all_rms, fnames, fluothreshrange
end
#=
    allplots = plotrobustness((all_p1, all_p2, all_p3), 
                    (all_rms1, all_rms2, all_rms3), fnames, fluothreshrange; 
                    fluothresh_default=fracabove_default)
    
    # Display and save the resulting plots
    ppp, pp2, pp3 = allplots

    param_names = ["kc", "t0"]
    for i in 1:1:2
        nm = param_names[i]
        fname = joinpath(plotdir, "paramfits_$(nm)_vs_threshold.pdf")
        display(ppp[i])
        pl.savefig(ppp[i], fname)
    end

    fname = joinpath(plotdir, "onehit_vs_twohit_fitRMS.pdf")
    display(pp2)
    if dosave
        pl.savefig(pp2, fname)
    end

    fname = joinpath(plotdir, "best12_vs_3-hit_fitRMS.pdf")
    display(pp3)
    if dosave
        pl.savefig(pp3, fname)
    end
    return 0
end

=#

In [ ]:

using Statistics: mean

"""
Clip excessive, artifactual RMS errors for plotting on log scale
"""
function cliprms!(rms; log10bound=4)
    log_rms = log10.(rms)
    mean_log_rms = mean(log_rms)
    wh = findall(log_rms .<= (mean_log_rms - log10bound))
    rms[wh] .= 10.0^mean_log_rms
    return rms
end


"""
Function making plots for the plot showing vs against fluorescence threshold. 
Update: make k and t0 separate panels, we only show k because the choice of range
of fluorescence thresholds is based on it, while t0 is noisy because of individual
tracks that pop in and out of the active set as the threshold changes. 
"""
function plotrobustness(fits123, rms123, allnames, 
            fracrange; fluothresh_default=0.5)
    # Unwrap arguments
    fits1, fits2, fits3 = fits123
    rms1, rms2, rms3 = rms123

    shortnames, lblconcs = cleannames(allnames)
    n_dsets = length(allnames)

    # Filter to only the datasets at 0.01pM and 0.1pM concentrations
    selected = findall(x -> x == 0.01 || x == 0.1, lblconcs)
    if isempty(selected)
        error("plotrobustness: no 0.01pM or 0.1pM datasets found in allnames")
    end
    sel_names = shortnames[selected]

    # Extract parameter indices for the selected datasets
    kappa_indices = 1 .+ selected
    t0_indices = 1 .+ n_dsets .+ selected

    # Use magma colors for the low-concentration datasets, matching their rank among all datasets.
    all_magma = pl.cgrad(:magma, n_dsets-2, categorical=true)
    sort_idx = sortperm(lblconcs)
    inverse_rank = zeros(Int, n_dsets)
    for (rank, idx) in enumerate(sort_idx)
        inverse_rank[idx] = rank
    end
    linecolors = [all_magma[inverse_rank[i]] for i in selected]

    # First plot: kappa and t0 across threshold choices, per dataset and model
    fracrangep = fracrange .+ 1.0
    p_kappa = pl.plot(layout=(1, 1), left_margin=6mm)
    p_t0 = pl.plot(layout=(1, 1), left_margin=6mm)
    for (sidx, dataset_idx) in enumerate(selected)
        lbl = sel_names[sidx]
        pl.plot!(p_kappa, fracrangep, fits1[:, kappa_indices[sidx]]; 
                label=lbl, color=linecolors[sidx], lw=2)
        pl.plot!(p_kappa, fracrangep, fits2[:, kappa_indices[sidx]]; 
                label=nothing, color=linecolors[sidx], linestyle=:dash, lw=2)
        pl.plot!(p_kappa, fracrangep, fits3[:, kappa_indices[sidx]]; 
                label=nothing, color=linecolors[sidx], linestyle=:dot, lw=2)

        pl.plot!(p_t0, fracrangep, fits1[:, t0_indices[sidx]]; 
                label=lbl, color=linecolors[sidx], lw=2)
        pl.plot!(p_t0, fracrangep, fits2[:, t0_indices[sidx]]; 
                label=nothing, color=linecolors[sidx], linestyle=:dash, lw=2)
        pl.plot!(p_t0, fracrangep, fits3[:, t0_indices[sidx]]; 
                label=nothing, color=linecolors[sidx], linestyle=:dot, lw=2)
    end

    # Add legend entries describing the model styles
    pl.plot!(p_kappa, [NaN, NaN], [NaN, NaN]; label="One-hit", color=:black, linestyle=:solid)
    pl.plot!(p_kappa, [NaN, NaN], [NaN, NaN]; label="Two-hit", color=:black, linestyle=:dash)
    pl.plot!(p_kappa, [NaN, NaN], [NaN, NaN]; label="Three-hit", color=:black, linestyle=:dot)

    for p in (p_kappa, p_t0)
        pl.vline!(p, [fluothresh_default+1.0]; linestyle=:dash, color=:black, label=nothing)
    end

    pl.xlabel!(p_kappa, raw"Fluorescence threshold factor $f$")
    pl.ylabel!(p_kappa, raw"Activation rate $\kappa$ (s$^\mathrm{-1}$)")
    pl.plot!(p_kappa; yscale=:log10, minorgrid=true, fgcolor_minorgrid=:white,
            legend=:topright)

    pl.xlabel!(p_t0, raw"Fluorescence threshold factor $f$")
    pl.ylabel!(p_t0, raw"Start time $t_0$ (s)")
    pl.plot!(p_t0; legend=:topleft, legend_title="Dataset")

    # Shade area where fits are reasonable
    good_frac_range = [0.25, 1.2]
    pl.vspan!(p_kappa, good_frac_range.+1.0, color=:grey, alpha=0.3, label=nothing)
    pl.vspan!(p_t0, good_frac_range.+1.0, color=:grey, alpha=0.3, label=nothing)
    pl.annotate!(p_kappa, 1.3, maximum(fits3[:, kappa_indices])*0.85, 
        ("Optimal region", :grey, :bottom, :left))

    # Final plots: RMS error of linear vs quadratic fits
    # and RMS error of 3-step vs best of 1 or 2 steps,
    # across various threshold fractions, for each file. 
    rms1_sel = copy(rms1[:, selected])
    rms2_sel = copy(rms2[:, selected])
    rms3_sel = copy(rms3[:, selected])

    # Remove points where threshold is too small and many trajectories start
    # already active, which is typically factor <= 0.4
    mask = (good_frac_range[1] .<= fracrange .<= good_frac_range[2])
    npoints = length(fracrange)
    skp = 2
    rms1_sel = rms1_sel[mask, :][1:skp:end, :]
    rms2_sel = rms2_sel[mask, :][1:skp:end, :]
    rms3_sel = rms3_sel[mask, :][1:skp:end, :]
    # For excessively small RMS, set them to NaNs
    rms1_sel = cliprms!(rms1_sel)
    rms2_sel = cliprms!(rms2_sel)
    rms3_sel = cliprms!(rms3_sel)

    msmin, msmax = 3.0, 7.0
    npts = size(rms1_sel)[1]
    msizes = collect(LinRange(msmin, msmax, npts))

    ## Plot of one-hit vs two-hit RMS error
    pp2 = pl.plot(layout=(1, 1), size=(550, 450), bottom_margin=2mm,
            legend=false, legendfontsize=9)
    plotlimits = [1e-2, max(maximum(rms1_sel), maximum(rms2_sel), maximum(rms3_sel))*1.5]
    pl.plot!(pp2, plotlimits, plotlimits; color=:black, 
                linestyle=:dot, label=nothing)
    for i in 1:length(selected)
        pl.scatter!(pp2, rms1_sel[:, i], rms2_sel[:, i]; 
                color=linecolors[i], ms=msizes, label=sel_names[i])
    end

    # Hand-build a legend by plotting and labeling next to it
    # Stupidest thing I had to do in Julia because Plots does not give control
    # over legend entries and does not reproduce marker sizes in them
    # Switch to makie in future projects, it's as easy to do there as in Matplotlib.
    # Grey rectangle behind, x, y: position of bottom left corner
    logrectangle(w, h, x, y) = pl.Shape(
        exp.(log(10) .* (x .+ [0,w,w,0])), 
        exp.(log(10) .* (y .+ [0,0,h,h])))

    function customlegend(pp, logrmslims, ms_sel, frac_sel)
        logrmsx = 0.64 * (logrmslims[2]-logrmslims[1]) + logrmslims[1]
        logrmsy = 0.25 * (logrmslims[2]-logrmslims[1]) + logrmslims[1]
        dx, dy = 0.15, 0.07

        rect1 = logrectangle(0.71, 0.45, logrmsx-0.35*dx, logrmsy-length(ms_sel)*dy)
        pl.plot!(pp, rect1, lw=0.0, fillalpha=0.3, 
            color=:grey, linecolor=nothing, label=nothing)

        # Legend title
        pl.annotate!(pp, 10^(logrmsx), 10^(logrmsy+0.7*dy), 
            ("Fluorescence\nthreshold factor " * raw"$f$", :black, :left, :bottom))
        
        for i in 1:1:length(ms_sel)
            # Marker for lowest frac range
            pl.scatter!(pp, [10^logrmsx], [10^(logrmsy-(i-1)*dy)], 
                color=:black, ms=ms_sel[i], label=nothing)
            # Label of that marker, next to it
            pl.annotate!(pp, 10^(logrmsx+dx), 10^(logrmsy-(i-1)*dy), 
                ((@sprintf "%.1f" frac_sel[i]+1.0), :black, :left, :center))
        end
    end

    # Add custom legend
    ms_inter = collect(LinRange(msmin, msmax, npts))[div(npts, 2)]
    frac_inter = fracrange[div(npts, 2)]
    ms_selection = [msmin, ms_inter, msmax]
    frac_selection = [good_frac_range[1], frac_inter, good_frac_range[2]]
    logrmslimit = log10.(plotlimits)
    customlegend(pp2, logrmslimit, ms_selection, frac_selection)

    pl.xlabel!(pp2, "RMS error for one-hit model")
    pl.ylabel!(pp2, "RMS error for two-hit model")
    pl.plot!(pp2; xscale=:log10, yscale=:log10, xlims=plotlimits, ylims=plotlimits,
        minorgrid=true, fgcolor_minorgrid=:white)

    pct_one_better = sum(rms1_sel .<= rms2_sel) / length(rms1_sel) * 100.0
    lbl_one = (@sprintf "%.1f" pct_one_better) * " %"
    lbl_two = (@sprintf "%.1f" (100.0 - pct_one_better)) * " %"
    pl.annotate!(pp2, plotlimits[1]*1.1, plotlimits[2]*0.9, 
        ("One-hit is better: " * lbl_one, :black, :left, :top))
    pl.annotate!(pp2, plotlimits[2]*0.9, plotlimits[1]*1.1, 
        ("Two-hit is better: " * lbl_two, :black, :right, :bottom))

    
    ## Plot of best(one-hit, two-hit) vs three-hit RMS error
    best12_rms = min.(rms1_sel, rms2_sel)
    pp3 = pl.plot(layout=(1, 1), size=(550, 450), bottom_margin=2mm,
            legend=false, minorgrid=true, fgcolor_minorgrid=:white)
    pl.plot!(pp3, plotlimits, plotlimits; color=:black, 
            linestyle=:dot, label=nothing)
    for i in 1:length(selected)
        pl.scatter!(pp3, best12_rms[:, i], rms3_sel[:, i]; 
                color=linecolors[i], ms=msizes, label=sel_names[i])
    end

    customlegend(pp3, logrmslimit, ms_selection, frac_selection)
    pl.xlabel!(pp3, "RMS error for one or two-hit model")
    pl.ylabel!(pp3, "RMS error for three-hit model")
    pl.plot!(pp3; xscale=:log10, yscale=:log10, 
        xlims=plotlimits, ylims=plotlimits,
        minorgrid=true, fgcolor_minorgrid=:white)

    pct_three_better = sum(rms3_sel .<= best12_rms) / length(best12_rms) * 100.0
    lbl_best12 = (@sprintf "%.1f" (100.0 - pct_three_better)) * " %"
    lbl_three = (@sprintf "%.1f" pct_three_better) * " %"
    pl.annotate!(pp3, plotlimits[1]*1.1, plotlimits[2]*0.9, 
        ("One- or two-hit is better: " * lbl_best12, :black, :left, :top))
    pl.annotate!(pp3, plotlimits[2]*0.9, plotlimits[1]*1.1, 
        ("Three-hit is better: " * lbl_three, :black, :right, :bottom))

    ppp = [p_kappa, p_t0]
    return [ppp, pp2, pp3]
end


In [ ]:
#all_params, all_rms, fnames, fluothreshrange = robustness_vs_thresh_main();

allplots = plotrobustness(all_params, all_rms, fnames, fluothreshrange;
                    fluothresh_default=default_fluo_factor)
    
# Display and save the resulting plots
ppp, pp2, pp3 = allplots
display(ppp[1])
display(pp2)
display(pp3)
#=

param_names = ["kc", "t0"]
for i in 1:1:2
    nm = param_names[i]
    fname = joinpath(plotdir, "paramfits_$(nm)_vs_threshold.pdf")
    display(ppp[i])
    pl.savefig(ppp[i], fname)
end

fname = joinpath(plotdir, "onehit_vs_twohit_fitRMS.pdf")
display(pp2)
if dosave
    pl.savefig(pp2, fname)
end

fname = joinpath(plotdir, "best12_vs_3-hit_fitRMS.pdf")
display(pp3)
if dosave
    pl.savefig(pp3, fname)
end
=#